# CS-245 Machine Learning Project
## Phase 4: Final Comparative Analysis

---

**Group Members:** 

Zayna Qasim

Sukaina Nasir

Hamna Shah


**Course:** CS-245 — Machine Learning

---

This is the final phase of the project. The goal here is to bring everything together — the three supervised models from Phase 2 and the clustering from Phase 3 — and compare them properly in one place.

The project guidelines specifically ask for:
- A comparison of all models
- Identification of the best-performing model
- Reasoning and insights behind the results

We do all of that here, along with a consolidated set of visualisations that are useful for the final report and presentation.

Make sure `processed_data.csv` is uploaded before running.


---
## Step 0 — Imports and Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.cluster import KMeans

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Metrics
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                              r2_score, silhouette_score)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f8f9fa",
    "axes.grid":        True,
    "grid.color":       "#e0e0e0",
    "grid.linestyle":   "--",
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})

PALETTE = ["#2E86AB", "#F18F01", "#A23B72", "#44BBA4", "#E94F37"]

print("Ready.")


---
## Step 1 — Load Data and Retrain All Models

We retrain all models here in one place so this notebook is fully self-contained. This also means the metrics you see here are consistent — everything is trained and evaluated on the same split.


In [ ]:
df = pd.read_csv("processed_data.csv")

FEATURES = ["area_marla", "bedroom", "bath", "type_enc", "purpose_enc", "city_enc"]
TARGET   = "price_lakh"

X = df[FEATURES]
y = df[TARGET]

# Same random state as Phase 2 — ensures identical split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaler for Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Dataset     : {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Train size  : {len(X_train):,}")
print(f"Test size   : {len(X_test):,}")


In [ ]:
# --- Linear Regression ---
print("Training Linear Regression...")
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# --- Random Forest ---
print("Training Random Forest...")
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# --- Gradient Boosting ---
print("Training Gradient Boosting...")
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1,
                                max_depth=4, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print("All models trained.")


---
## Step 2 — Consolidated Metrics Table

This is the main comparison. We compute RMSE, MAE, and R2 for all three supervised models on the same test set.

| Metric | What it measures |
|--------|-----------------|
| RMSE | Root Mean Squared Error — average error in Lakhs, penalises large errors more |
| MAE | Mean Absolute Error — average absolute error in Lakhs, easier to interpret directly |
| R2 | How much of the price variance is explained by the model (1.0 = perfect) |


In [ ]:
def get_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return round(rmse, 2), round(mae, 2), round(r2, 4)

rows = []
all_preds = {}

for name, preds in [
    ("Linear Regression",  y_pred_lr),
    ("Random Forest",      y_pred_rf),
    ("Gradient Boosting",  y_pred_gb),
]:
    rmse, mae, r2 = get_metrics(y_test, preds)
    rows.append({"Model": name, "RMSE (Lakhs)": rmse, "MAE (Lakhs)": mae, "R2 Score": r2})
    all_preds[name] = preds

results_df = pd.DataFrame(rows)

print("=" * 58)
print("SUPERVISED MODEL COMPARISON — TEST SET RESULTS")
print("=" * 58)
print(results_df.to_string(index=False))
print("=" * 58)
print()
print("Lower RMSE and MAE = better prediction accuracy")
print("Higher R2          = more variance in price explained")


In [ ]:
# Cross-validation scores for all models
print("Running 5-Fold Cross Validation...")
print()

cv_scores = {}

lr_cv = cross_val_score(LinearRegression(), X_train_scaled, y_train, cv=5, scoring="r2")
cv_scores["Linear Regression"] = lr_cv
print(f"Linear Regression  | CV R2: {lr_cv.round(3)} | Mean: {lr_cv.mean():.4f} | Std: {lr_cv.std():.4f}")

rf_cv = cross_val_score(RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
                         X_train, y_train, cv=5, scoring="r2")
cv_scores["Random Forest"] = rf_cv
print(f"Random Forest      | CV R2: {rf_cv.round(3)} | Mean: {rf_cv.mean():.4f} | Std: {rf_cv.std():.4f}")

gb_cv = cross_val_score(GradientBoostingRegressor(n_estimators=50, random_state=42),
                         X_train, y_train, cv=5, scoring="r2")
cv_scores["Gradient Boosting"] = gb_cv
print(f"Gradient Boosting  | CV R2: {gb_cv.round(3)} | Mean: {gb_cv.mean():.4f} | Std: {gb_cv.std():.4f}")

results_df["CV R2 Mean"] = [lr_cv.mean().round(4), rf_cv.mean().round(4), gb_cv.mean().round(4)]
results_df["CV R2 Std"]  = [lr_cv.std().round(4),  rf_cv.std().round(4),  gb_cv.std().round(4)]


---
## Step 3 — Unsupervised Model Summary

K-Means Clustering is evaluated separately since it is not a prediction task — there is no RMSE or R2. The main metric is the **Silhouette Score**.


In [ ]:
# Retrain clustering with K=3 (consistent with Phase 3 findings)
CLUSTER_FEATURES = ["area_marla", "bedroom", "bath", "price_lakh", "city_enc", "type_enc"]

X_cluster = df[CLUSTER_FEATURES]
cluster_scaler = StandardScaler()
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster_scaled)

sil_score = silhouette_score(X_cluster_scaled, cluster_labels)
df["cluster"] = cluster_labels

print("K-Means Clustering Summary")
print("-" * 35)
print(f"Number of clusters (K) : 3")
print(f"Silhouette Score       : {sil_score:.4f}")
print(f"Cluster distribution   :")
for cid, count in sorted(zip(*np.unique(cluster_labels, return_counts=True))):
    print(f"  Cluster {cid}  ->  {count:,} properties")

print()
print("Silhouette score interpretation:")
print("  0.71 - 1.00  ->  Strong structure")
print("  0.51 - 0.70  ->  Reasonable structure")
print("  0.26 - 0.50  ->  Weak but present structure")
print("  < 0.26       ->  No real structure found")


---
## Step 4 — Visualisations

### Plot 1: Full Model Comparison — RMSE, MAE, R2 Side by Side


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Supervised Model Comparison — All Metrics", fontsize=14, fontweight="bold")

model_names  = results_df["Model"]
short_names  = ["Linear
Regression", "Random
Forest", "Gradient
Boosting"]
bar_colors   = [PALETTE[0], PALETTE[1], PALETTE[2]]

metrics = [
    ("RMSE (Lakhs)", "RMSE   (lower is better)",  0),
    ("MAE (Lakhs)",  "MAE   (lower is better)",   1),
    ("R2 Score",     "R2 Score   (higher is better)", 2),
]

for col, (metric, title, ax_idx) in enumerate(metrics):
    ax = axes[ax_idx]
    bars = ax.bar(short_names, results_df[metric], color=bar_colors,
                  edgecolor="white", width=0.5)
    ax.set_title(title)
    ax.set_ylabel(metric)
    if metric == "R2 Score":
        ax.set_ylim(0, 1.1)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + (0.01 if metric == "R2 Score" else 1.5),
                str(val), ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("plot_comparison_metrics.png", dpi=150, bbox_inches="tight")
plt.show()


### Plot 2: Actual vs Predicted — All Three Models

This is the clearest way to see model quality visually. All points should sit on the red diagonal if the model is perfect.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Actual vs Predicted Property Prices (Lakhs)", fontsize=14, fontweight="bold")

model_data = [
    ("Linear Regression",  y_pred_lr, PALETTE[0]),
    ("Random Forest",      y_pred_rf, PALETTE[1]),
    ("Gradient Boosting",  y_pred_gb, PALETTE[2]),
]

for ax, (name, preds, color) in zip(axes, model_data):
    ax.scatter(y_test, preds, alpha=0.25, s=8, color=color, edgecolors="none")
    lo = min(y_test.min(), preds.min())
    hi = max(y_test.max(), preds.max())
    ax.plot([lo, hi], [lo, hi], color="red", linewidth=1.5,
            linestyle="--", label="Perfect prediction")
    r2 = r2_score(y_test, preds)
    ax.set_title(f"{name}\nR2 = {r2:.4f}")
    ax.set_xlabel("Actual Price (Lakhs)")
    ax.set_ylabel("Predicted Price (Lakhs)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("plot_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

print("Points closer to the red line = more accurate predictions.")


### Plot 3: Residual Analysis

Residuals (Actual - Predicted) should be randomly scattered around zero. Any clear pattern means the model is missing something systematic.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Residual Analysis — All Models", fontsize=14, fontweight="bold")

for col, (name, preds, color) in enumerate(model_data):
    residuals = y_test.values - preds

    # Residuals vs Predicted
    axes[0, col].scatter(preds, residuals, alpha=0.25, s=8,
                          color=color, edgecolors="none")
    axes[0, col].axhline(0, color="red", linewidth=1.5, linestyle="--")
    axes[0, col].set_title(f"{name} — Residuals vs Predicted")
    axes[0, col].set_xlabel("Predicted Price (Lakhs)")
    axes[0, col].set_ylabel("Residual")

    # Residual distribution
    axes[1, col].hist(residuals, bins=60, color=color,
                       edgecolor="white", alpha=0.85)
    axes[1, col].axvline(0, color="red", linewidth=1.5, linestyle="--")
    axes[1, col].set_title(f"{name} — Residual Distribution")
    axes[1, col].set_xlabel("Residual (Actual - Predicted)")
    axes[1, col].set_ylabel("Count")

plt.tight_layout()
plt.savefig("plot_residual_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print("A bell-shaped residual distribution centered at 0 is ideal.")
print("Skewed or wide distributions indicate the model is missing patterns.")


### Plot 4: Learning Curves

Learning curves show how model performance changes as we increase the training set size. This tells us whether a model would benefit from more data or whether it is already saturated.

- If training score and validation score are both high and converging — good model.
- Large gap between them — model is overfitting.
- Both scores low — model is underfitting.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Learning Curves", fontsize=14, fontweight="bold")

lc_models = [
    ("Linear Regression",
     LinearRegression(), X_train_scaled, PALETTE[0]),
    ("Random Forest",
     RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1), X_train, PALETTE[1]),
    ("Gradient Boosting",
     GradientBoostingRegressor(n_estimators=50, random_state=42), X_train, PALETTE[2]),
]

for ax, (name, model, X_lc, color) in zip(axes, lc_models):
    print(f"  Computing learning curve for {name}...")
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_lc, y_train,
        cv=3, scoring="r2",
        train_sizes=np.linspace(0.1, 1.0, 8),
        n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    val_mean   = val_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_std    = val_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, label="Training score",
            color=color, linewidth=2, marker="o")
    ax.plot(train_sizes, val_mean, label="Validation score",
            color=color, linewidth=2, marker="s", linestyle="--")
    ax.fill_between(train_sizes, train_mean - train_std,
                     train_mean + train_std, alpha=0.1, color=color)
    ax.fill_between(train_sizes, val_mean - val_std,
                     val_mean + val_std, alpha=0.1, color=color)
    ax.set_title(name)
    ax.set_xlabel("Training Set Size")
    ax.set_ylabel("R2 Score")
    ax.set_ylim(-0.1, 1.1)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("plot_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()


### Plot 5: Feature Importance Comparison

Random Forest and Gradient Boosting both give us feature importance scores. We compare them side by side to see if they agree on what drives property prices.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Feature Importance — Random Forest vs Gradient Boosting",
             fontsize=14, fontweight="bold")

for ax, (model, name, color) in zip(axes, [
    (rf, "Random Forest",      PALETTE[1]),
    (gb, "Gradient Boosting",  PALETTE[2]),
]):
    imp = pd.DataFrame({
        "Feature":    FEATURES,
        "Importance": model.feature_importances_
    }).sort_values("Importance")

    bars = ax.barh(imp["Feature"], imp["Importance"],
                   color=color, edgecolor="white", alpha=0.85)
    ax.set_title(name)
    ax.set_xlabel("Importance Score")
    for bar, val in zip(bars, imp["Importance"]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("plot_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("If both models agree on the most important features, it gives us more")
print("confidence that those features genuinely drive property prices in this dataset.")


### Plot 6: Cross-Validation Score Distribution

Box plots of the 5-fold CV R2 scores. A tighter box means more consistent performance across different subsets of the data.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

cv_data   = [cv_scores["Linear Regression"],
             cv_scores["Random Forest"],
             cv_scores["Gradient Boosting"]]
cv_labels = ["Linear
Regression", "Random
Forest", "Gradient
Boosting"]

bp = ax.boxplot(cv_data, patch_artist=True, widths=0.45,
                medianprops={"color": "white", "linewidth": 2})

for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_xticklabels(cv_labels)
ax.set_ylabel("R2 Score")
ax.set_title("5-Fold Cross-Validation R2 Score Distribution")
ax.set_ylim(0, 1.05)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("plot_cv_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


### Plot 7: Prediction Error Distribution

How are prediction errors distributed across models? This is a direct look at where each model struggles.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for name, preds, color in model_data:
    errors = np.abs(y_test.values - preds)
    ax.hist(errors, bins=80, alpha=0.55, color=color,
            label=f"{name}  (median error = {np.median(errors):.1f} Lakhs)",
            edgecolor="none")

ax.set_xlabel("Absolute Prediction Error (Lakhs)")
ax.set_ylabel("Number of Properties")
ax.set_title("Absolute Prediction Error Distribution — All Models")
ax.legend()
ax.set_xlim(left=0)

plt.tight_layout()
plt.savefig("plot_error_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("A distribution shifted further left means the model makes smaller errors more often.")


### Plot 8: Cluster Profiles — Final Summary View

A summary of the three property clusters found in Phase 3, showing how each cluster differs across key features.


In [ ]:
cluster_summary = df.groupby("cluster")[["area_marla", "bedroom", "bath", "price_lakh"]].mean().round(2)
cluster_summary.index = [f"Cluster {i}" for i in cluster_summary.index]

# Sort by price
cluster_summary = cluster_summary.sort_values("price_lakh")

# Assign segment labels
segment_labels = ["Budget", "Mid-Range", "Luxury"]
cluster_summary.insert(0, "Segment", segment_labels[:len(cluster_summary)])

print("Cluster Profiles Summary:")
print(cluster_summary.to_string())
print(f"\nSilhouette Score: {sil_score:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Cluster Profiles — Mean Feature Values per Segment",
             fontsize=13, fontweight="bold")

feat_cols = ["area_marla", "bedroom", "bath", "price_lakh"]
feat_labels = ["Area (Marla)", "Bedrooms", "Bathrooms", "Price (Lakhs)"]
seg_colors = [PALETTE[3], PALETTE[1], PALETTE[2]]

for ax, col, label in zip(axes, feat_cols, feat_labels):
    bars = ax.bar(cluster_summary["Segment"],
                  cluster_summary[col],
                  color=seg_colors[:len(cluster_summary)],
                  edgecolor="white", width=0.5)
    ax.set_title(label)
    ax.set_ylabel(label)
    for bar, val in zip(bars, cluster_summary[col]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                f"{val:.1f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("plot_cluster_summary.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Step 5 — Final Discussion and Model Selection

### Supervised Models

**Linear Regression** was the weakest model with an R2 of 0.5636. It explains around 56% of the variance in property prices, which is decent for a baseline but not good enough for real-world use. The main issue is that property prices in Pakistan don't follow a simple linear pattern — the relationship between features like city and price is highly non-linear. A house in DHA Lahore isn't just a constant offset — the entire pricing dynamic changes.

**Random Forest** was the best-performing model. R2 of 0.8085 means it explains over 80% of the variance in prices. The RMSE of 124.82 Lakhs and MAE of 68.04 Lakhs are considerably better than Linear Regression. The cross-validation scores were also consistent across folds, which means the model generalises well and isn't just overfitting to one particular train/test split.

**Gradient Boosting** was a close second with an R2 of 0.7947. Slightly behind Random Forest in all three metrics but still a strong model. The learning curves show it may benefit from more training data or hyperparameter tuning, which we didn't do here due to time constraints.

### Best Model: Random Forest

We select Random Forest as the final recommended model based on:
- Highest test set R2 (0.8085)
- Lowest RMSE (124.82 Lakhs) and MAE (68.04 Lakhs)
- Stable cross-validation scores with low standard deviation
- Feature importance is interpretable — area and city dominate, which aligns with our domain knowledge

### Unsupervised Model

K-Means with K=3 produced a silhouette score of 0.4313, which falls in the "weak but present structure" range. This is acceptable for a real-world dataset like this. Property prices are continuous and overlapping — there are no truly clean boundaries between budget, mid-range, and luxury segments. The clusters still provide useful segmentation for business purposes even if they're not perfectly separated in the feature space.

The three clusters roughly correspond to:
- Budget properties — smaller area, fewer bedrooms, lower price
- Mid-range — average across all features, the majority of listings
- Luxury — larger area, more bedrooms and bathrooms, significantly higher price

### Feature Importance Findings

Both Random Forest and Gradient Boosting agree that `area_marla` and `city_enc` are the most important predictors. This makes complete sense — in Pakistan, location is a primary driver of property value. A 5 Marla house in DHA Islamabad can cost several times more than the same size house in a smaller city.

`bedroom` and `bath` contribute but matter less once area and city are accounted for, which also aligns with how people typically price property.

### Limitations

- The dataset has some noise from web scraping — some listed prices may not reflect actual transaction prices
- We used Label Encoding for city which assigns arbitrary integers — for Linear Regression especially, this is not ideal. One-hot encoding would be more appropriate for a linear model
- The model has not been tested on properties outside the cities in the dataset
- We did not tune hyperparameters for Random Forest or Gradient Boosting — tuning could improve both models further

---
All plots have been saved and can be used directly in the final report and presentation.


---
## Step 6 — Complete Results Summary

A single consolidated table covering all models for inclusion in the report.


In [ ]:
print("=" * 70)
print("COMPLETE MODEL COMPARISON SUMMARY")
print("=" * 70)
print()
print("SUPERVISED MODELS (Regression Task: Predict Property Price)")
print("-" * 70)
print(f"{'Model':<22} {'RMSE':>10} {'MAE':>10} {'R2':>10} {'CV R2 Mean':>12}")
print("-" * 70)

for _, row in results_df.iterrows():
    print(f"{row['Model']:<22} {row['RMSE (Lakhs)']:>10} {row['MAE (Lakhs)']:>10} "
          f"{row['R2 Score']:>10} {row['CV R2 Mean']:>12}")

print("-" * 70)
print()
print("UNSUPERVISED MODEL (Clustering Task: Segment Properties)")
print("-" * 70)
print(f"{'Model':<22} {'K':>10} {'Silhouette':>12} {'Segments':>15}")
print("-" * 70)
print(f"{'K-Means':<22} {3:>10} {sil_score:>12.4f} {'Budget/Mid/Luxury':>15}")
print("-" * 70)
print()
print("Best supervised model : Random Forest  (R2 = 0.8085)")
print("Clustering quality    : Weak-moderate structure (Silhouette = 0.43)")
print("=" * 70)
